# Feature Screening

## Set-up

In [ ]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

project = pd.read_csv('../datasets/combined_activity_data.csv')
project['datetime'] = pd.to_datetime(project['datetime'])
display(project.head())
display(project.info())

In [ ]:
# Create Date and Hour Features for futher feature engineering
project['date'] = project['datetime'].dt.date
project['hour'] = project['datetime'].dt.hour
day_row = []

# Create day-level features for each participant
for (id, day), group in project.groupby(['id', 'date']):
    group = group.sort_values('datetime')
    steps = group['steps'].to_numpy()
    active = (steps > 0).astype(int)

    day_data = {
        # Metadata
        'id': id,
        'date': day,
        'label': group['label'].iloc[0],
        'source': group['source'].iloc[0],
        
        # Standard Step Features
        'total_steps': group['steps'].sum(),
        'mean_steps': group['steps'].mean(),
        'std_steps': group['steps'].std(ddof = 0), # Each day fully observed, so population std
        'cv_steps': group['steps'].std(ddof = 0) / group['steps'].mean() if group['steps'].mean() > 0 else 0,
        'max_steps': group['steps'].max(),
        'min_steps': group['steps'].min(),

        # Look at activity patterns across the day
        # Basically, how active are they vs not, and when are they actually active
        'active_hours': active.sum(),
        'sedentary_hours': (active == 0).sum(),
        'active_sedentary_ratio': active.sum() / (active == 0).sum() if (active == 0).sum() > 0 else active.sum(),
        'early_morning_steps': group.loc[group['hour'].between(0, 5), 'steps'].sum(),
        'morning_steps': group.loc[group['hour'].between(6, 11), 'steps'].sum(),
        'afternoon_steps': group.loc[group['hour'].between(12, 17), 'steps'].sum(),
        'evening_steps': group.loc[group['hour'].between(18, 23), 'steps'].sum(),
    }
    day_row.append(day_data)

# Create a new DataFrame with the engineered day-level features
project_day = pd.DataFrame(day_row)
project_day['date'] = pd.to_datetime(project_day['date'])
display(project_day.head())
display(project_day.info())

In [ ]:
# Distribution of features
feature_col = [
    'total_steps', 'mean_steps', 'std_steps',
    'cv_steps', 'max_steps', 'min_steps',

    'active_hours', 'sedentary_hours', 'active_sedentary_ratio',
    'early_morning_steps', 'morning_steps',
    'afternoon_steps', 'evening_steps'
]

project_day[feature_col].hist(bins = 20, figsize = (16, 12))
plt.tight_layout()
plt.savefig('../output/feature_distribution.png', dpi = 300)

In [ ]:
# Drop Correlated Features
vlag = sns.color_palette('vlag', as_cmap = True)
corr = project_day[feature_col].corr().abs()
plt.figure(figsize = (20, 14))
sns.heatmap(corr, annot = True, fmt = '.2f', cmap = vlag)
plt.savefig('../output/heatmap.png', dpi = 300)
display(corr)

feature_col = [
    'total_steps', 'cv_steps', 'active_hours', 
    'early_morning_steps', 'morning_steps',
    'afternoon_steps', 'evening_steps'
]


In [ ]:
# Prep Data 
X = project_day[feature_col].values
y = project_day['label'].values
groups = project_day['id'].values
gkf = GroupKFold(n_splits = 5)
folds = []

# 5-fold CV
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups = groups), start = 1):
    # Train-Test Split
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Get # MS & Healthy in Training
    healthy = np.sum(y_train == 0)
    ms = np.sum(y_train == 1)
    total = healthy + ms
    class_weight = {
    0: total / (2 * healthy),
    1: total / (2 * ms)
    } # Build inverse-frequency weight to help deal with class imbalance

    # Right-skewed -> RobustScaler()
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Fit Model
    log_mod = LogisticRegression(
        class_weight = class_weight,
        max_iter = 1000,
        random_state = 223
    )
    log_mod.fit(X_train_scaled, y_train)

    # Prediction
    y_pred = log_mod.predict(X_test_scaled) # 0/1
    y_prob = log_mod.predict_proba(X_test_scaled)[:, 1] # P(MS)

    # Model Diagnostic
    folds.append({
        'fold' : fold,
        'auc' : roc_auc_score(y_test, y_prob),
        'precision' : precision_score(y_test, y_pred, zero_division = 0), # zero_division -> Set value 0 when occurs
        'recall' : recall_score(y_test, y_pred, zero_division = 0),
        'f1' : f1_score(y_test, y_pred, zero_division = 0)
    })

# Display Results and Output Averages
kfold_res = pd.DataFrame(folds)

new_row = pd.DataFrame([{
    'fold' : "Mean", 
    'auc' : kfold_res['auc'].mean(), 
    'precision' : kfold_res['precision'].mean(),
    'recall' : kfold_res['recall'].mean(),
    'f1' : kfold_res['f1'].mean()
    }])
kfold_res = pd.concat([kfold_res, new_row], ignore_index = True)
display(kfold_res)
kfold_res.to_csv('../output/kfold_results.csv')

In [ ]:
# Final Model
X = project_day[feature_col].values
y = project_day['label'].values
healthy = np.sum(y == 0)
ms = np.sum(y == 1)
total = healthy + ms
class_weight = {
    0: total / (2 * healthy),
    1: total / (2 * ms)
    }
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
log_final = LogisticRegression(
    class_weight = class_weight,
    max_iter = 1000,
    random_state = 223
    )
result = log_final.fit(X_scaled, y)

model_coef = pd.DataFrame({
    "Feature" : feature_col,
    "Coefficient" : result.coef_[0],
    "ABS_Coefficient" : np.abs(result.coef_[0]),
    "Odds_Ratio" : np.exp(result.coef_[0])
}).sort_values("ABS_Coefficient", ascending = False)

model_coef.to_csv('../output/feature_coefficients.csv', index = False)
display(model_coef)

In [ ]:
# Shap
explainer = shap.LinearExplainer(log_final, X_scaled)
shap_values = explainer.shap_values(X_scaled)
shap.summary_plot(
    shap_values,
    X_scaled, 
    feature_names = feature_col ,
    show = False
    )
plt.tight_layout()
plt.savefig('../output/shap_sum_plot.png', dpi = 300)
plt.show()
project_day.to_csv('../datasets/project_day.csv')